https://lamport.azurewebsites.net/tla/tutorial/intro.html

EXPORT CLASSPATH=tla2tools.jar
java tla2sany.SANY -help  # The TLA⁺ parser
java tlc2.TLC -help       # The TLA⁺ model checker
java tlc2.REPL            # Enter the TLA⁺ REPL
java pcal.trans -help     # The PlusCal-to-TLA⁺ translator
java tla2tex.TLA -help    # The TLA⁺-to-LaTeX translator
java tla2sany.xml.XMLExporter -help # Export TLA⁺ parse tree as XML


In [24]:
%%file /tmp/mytest.tla
------ MODULE mytest -----------

Foo == 3

(*
--algorithm AnyName {
   variable x = {"a","b"} ;  
   {  
     x := x \cup {"c"} ;
     print x
   }
}
*)

===============================

Overwriting /tmp/mytest.tla


-o makes it not validate against a schema that is no longer being served. It is not output. Insanity. Complete derangement.

In [25]:
! java -cp ~/Downloads/tla2tools.jar tla2sany.xml.XMLExporter -o /tmp/mytest.tla

Semantic processing of module mytest
<?xml version="1.0" encoding="UTF-8" standalone="no"?><modules><RootModule>mytest</RootModule><context><entry><UID>154</UID><ModuleNode><location><column><begin>1</begin><end>31</end></column><line><begin>1</begin><end>15</end></line><filename>mytest</filename></location><uniquename>mytest</uniquename><UserDefinedOpKindRef><UID>156</UID></UserDefinedOpKindRef></ModuleNode></entry><entry><UID>156</UID><UserDefinedOpKind><location><column><begin>1</begin><end>8</end></column><line><begin>3</begin><end>3</end></line><filename>mytest</filename></location><level>0</level><uniquename>Foo</uniquename><arity>0</arity><body><NumeralNode><location><column><begin>8</begin><end>8</end></column><line><begin>3</begin><end>3</end></line><filename>mytest</filename></location><level>0</level><IntValue>3</IntValue></NumeralNode></body><params/></UserDefinedOpKind></entry></context><ModuleNodeRef><UID>154</UID></ModuleNodeRef></modules>

In [ ]:
! java -cp ~/Downloads/tla2tools.jar pcal.trans /tmp/mytest.tla && cat /tmp/mytest.tla && java -cp ~/Downloads/tla2tools.jar 

pcal.trans Version 1.11 of 31 December 2020
Labels added.
Parsing completed.
Translation completed.
New file /tmp/mytest.tla written.
New file /tmp/mytest.cfg written.
------ MODULE mytest -----------

(*
--algorithm AnyName {
   variable x = {"a","b"} ;  
   {  
     x := x \cup {"c"} ;
     print x
   }
}
*)
\* BEGIN TRANSLATION (chksum(pcal) = "173b0586" /\ chksum(tla) = "461a4402")
VARIABLES x, pc

vars == << x, pc >>

Init == (* Global variables *)
        /\ x = {"a","b"}
        /\ pc = "Lbl_1"

Lbl_1 == /\ pc = "Lbl_1"
         /\ x' = (x \cup {"c"})
         /\ PrintT(x')
         /\ pc' = "Done"

(* Allow infinite stuttering to prevent deadlock on termination. *)
Terminating == pc = "Done" /\ UNCHANGED vars

Next == Lbl_1
           \/ Terminating

Spec == Init /\ [][Next]_vars

Termination == <>(pc = "Done")

\* END TRANSLATION 



In [21]:
import ast


def expr(e : ast.expr):
    match e:
        case ast.Name(id=name):
            return name
        case ast.Constant(value=value):
            return repr(value)
        case _:
            raise ValueError(f"Unsupported expression: {ast.dump(e)}")

def stmt(e : ast.stmt):
    match e:
        case ast.Assign(targets=[ast.Name(id=var)], value=value):
            return f"{var} := {expr(value)}"
        case ast.Expr(value=ast.Call(func=ast.Name(id='print'), args=[arg])):
            return f"print {expr(arg)}"
        case _:
            raise ValueError(f"Unsupported statement: {ast.dump(e)}")


def to_pluscal(mod):
    match mod:
        case ast.Module(body=body):
            return "{\n" + "\n".join(stmt(e) for e in body) + "\n}"
        case _:
            raise ValueError("Expected ast.Module")
    
print(to_pluscal(ast.parse("""
x = 3
print(x)
""")))

{
x := 3
print x
}


In [30]:
%%file /tmp/HourClock.tla
---- MODULE HourClock ----
EXTENDS Naturals

VARIABLE hr

HCini == hr \in (1 .. 12)

HCnxt == hr' = IF hr = 12 THEN 1 ELSE hr + 1
          (* Alternately expressed as: hr' = (hr % 12) + 1 *)

HC == HCini /\ [][HCnxt]_hr
==========================

Writing /tmp/HourClock.tla


https://github.com/tlaplus/tlaplus/blob/master/tlatools/org.lamport.tlatools/src/tla2sany/xml/sany.xsd

In [ ]:
import subprocess

xml = subprocess.run(["java", "-cp", "/home/philip/Downloads/tla2tools.jar", "tla2sany.xml.XMLExporter", "-o", "/tmp/HourClock.tla"], check=True, capture_output=True).stdout

import xml.etree.ElementTree as ET
mods = ET.fromstring(xml)
mods.tag
mods.attrib
mods[1][0]

ET.indent(mods, space="  ")
ET.tostring(mods, encoding="unicode")

# OpDeclNodeRef
# OpDeclNode
for e in mods.iter("OpDeclNode"): 
    print(ET.tostring(e, encoding="unicode"))


<OpDeclNode>
        <location>
          <column>
            <begin>10</begin>
            <end>11</end>
          </column>
          <line>
            <begin>4</begin>
            <end>4</end>
          </line>
          <filename>HourClock</filename>
        </location>
        <level>1</level>
        <uniquename>hr</uniquename>
        <arity>0</arity>
        <kind>3</kind>
      </OpDeclNode>
    


In [ ]:
import subprocess
from dataclasses import dataclass
import xml.etree.ElementTree as ET


@dataclass
class App:
    f: object
    args: list["App"]

    def __repr__(self):
        if not self.args:
            return f"{self.f}"
        else:
            return f"{self.f}({', '.join(map(str, self.args))})"


def tla_xml(filename):
    # https://github.com/tlaplus/tlaplus/blob/master/tlatools/org.lamport.tlatools/src/tla2sany/xml/sany.xsd
    return subprocess.run(
        [
            "java",
            "-cp",
            "/home/philip/Downloads/tla2tools.jar",
            "tla2sany.xml.XMLExporter",
            "-o",
            filename,
        ],
        check=True,
        capture_output=True,
    ).stdout


def tla_exprs(filename):
    mods = ET.fromstring(tla_xml(filename))

    by_uid = {}
    for entry in mods.find("context"):
        uid = entry.findtext("UID")
        node = next((c for c in entry if c.tag != "UID"), None)
        if uid is not None and node is not None:
            by_uid[uid] = node

    def ref_name(ref):
        uid = ref.findtext("UID")
        node = by_uid.get(uid)
        if node is None:
            return f"UID:{uid}"
        return node.findtext("uniquename") or node.tag

    def expr(node):
        if node.tag == "OpApplNode":
            ref = next(iter(node.find("operator")), None)
            f = ref_name(ref) if ref is not None else node.tag
            operands = node.find("operands")
            args = [] if operands is None else [expr(arg) for arg in operands]
            return App(f, args)
        if node.tag.endswith("Ref"):
            return App(ref_name(node), [])
        if node.tag == "NumeralNode":
            return App(int(node.findtext("IntValue")), [])
        return App(node.findtext("uniquename") or node.tag, [])

    res = {}
    for node in by_uid.values():
        if node.tag != "UserDefinedOpKind":
            continue
        if node.findtext("./location/filename") != mods.findtext("RootModule"):
            continue
        body = node.find("./body/*")
        if body is not None:
            res[node.findtext("uniquename")] = expr(body)
    return res


import z3


def to_z3(e: App, decls: dict[str, z3.FuncDeclRef], sort=None):
    if isinstance(e.f, int) and not e.args:
        assert sort is None or sort == z3.IntSort()
        return z3.IntVal(e.f)
    elif e.f in decls:
        f = decls[e.f]
        assert len(e.args) == f.arity()
        assert sort is None or sort == f.range()
        args = [to_z3(arg, decls, sort=f.domain(i)) for i, arg in enumerate(e.args)]
        return f(*args)
    elif e.f == "\\land":
        assert len(e.args) >= 2
        args = [to_z3(arg, decls, sort=z3.BoolSort()) for arg in e.args]
        return z3.And(*args)
    elif e.f == "\\lor":
        assert len(e.args) >= 2
        args = [to_z3(arg, decls, sort=z3.BoolSort()) for arg in e.args]
        return z3.Or(*args)
    elif e.f == "=":
        assert len(e.args) == 2
        args = [to_z3(arg, decls) for arg in e.args]
        return args[0] == args[1]
    elif e.f == "..":
        assert len(e.args) == 2
        args = [to_z3(arg, decls, sort=z3.IntSort()) for arg in e.args]
        x = z3.FreshConst(z3.IntSort(), prefix="x")
        return z3.Lambda(x, z3.And(args[0] <= x, x <= args[1]))
    elif e.f == "\\in":
        x = to_z3(e.args[0], decls)
        s = to_z3(e.args[1], decls, sort=z3.SetSort(x.sort()))
        return z3.IsMember(x, s)
    elif e.f == "'":
        assert len(e.args) == 1
        x = to_z3(e.args[0], decls, sort=sort)
        return z3.Const(x.decl().name() + "'", x.sort())
    elif e.f == "$IfThenElse":
        assert len(e.args) == 3
        cond = to_z3(e.args[0], decls, sort=z3.BoolSort())
        then = to_z3(e.args[1], decls, sort=sort)
        else_ = to_z3(e.args[2], decls, sort=sort)
        return z3.If(cond, then, else_)
    elif e.f == "+":
        assert len(e.args) == 2
        args = [to_z3(arg, decls, sort=sort) for arg in e.args]
        return args[0] + args[1]
    elif e.f == "$SquareAct":
        return z3.Const("$SQUAREACT TODO", z3.BoolSort())
    elif e.f == "[]":
        return z3.Const("[] TODO", z3.BoolSort())
    # elif sort is not None:
    #    # fallback to uninterprted function
    #    args = [to_z3(arg, decls) for arg in e.args]
    #    f = z3.Function(e.f, *[arg.sort() for arg in args], sort)
    #    return f(*args)
    else:
        raise ValueError(f"Cannot convert {e} to z3 without sort")


if __name__ == "__main__":
    exprs = tla_exprs("/tmp/HourClock.tla")
    decls = {"hr": z3.Function("hr", z3.IntSort())}
    for name, e in exprs.items():
        print(f"{name} = {e}")
        ze = to_z3(e, decls, sort=None)
        print(decls)
        decls[name] = z3.Function(name, ze.sort())
        print(ze)


"""
Notes:
I could take variable declarations and implicitly raise all definitions in the module to take those vars and their primed versions as explicit params.

Bidirectional type inference on the cheap.
Anonymous keyed records like tuples. That'd be kind of nice.


"""
